# Exploratory Data Analysis

This notebook examines the PhonePe district and state data before opportunity scoring. The focus is on extracting decision-useful information about scale, concentration, growth quality, merchant penetration, category mix, stability, and analytical limitations.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

from phonepe_analytics.metrics import add_growth_metrics, add_ratio_metrics

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data' / 'processed'
CHARTS = ROOT / 'reports' / 'eda_charts'
CHARTS.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 60)

district = pd.read_csv(DATA / 'district_quarter.csv')
state = pd.read_csv(DATA / 'state_quarter.csv')
categories = pd.read_csv(DATA / 'state_transaction_categories.csv')
district = add_growth_metrics(add_ratio_metrics(district), ['state', 'district'])
state = add_growth_metrics(add_ratio_metrics(state), ['state'])

latest_period = int(district['period_id'].max())
latest = district[district['period_id'].eq(latest_period)].copy()
latest_state = state[state['period_id'].eq(latest_period)].copy()
latest_label = f"{int(latest['year'].max())} Q{int(latest['quarter'].max())}"

def insight(*lines):
    display(Markdown('**Insight**\n\n' + '\n'.join(f'- {line}' for line in lines)))

def save(name):
    plt.tight_layout()
    plt.savefig(CHARTS / name, dpi=160, bbox_inches='tight')
    plt.show()

def profile(frame, columns):
    rows = []
    for column in columns:
        series = frame[column].replace([np.inf, -np.inf], np.nan).dropna()
        q1, q3 = series.quantile([0.25, 0.75])
        iqr = q3 - q1
        rows.append({'variable': column, 'count': len(series), 'missing': frame[column].isna().sum(), 'mean': series.mean(), 'median': series.median(), 'std': series.std(), 'p10': series.quantile(0.10), 'p25': q1, 'p75': q3, 'p90': series.quantile(0.90), 'p95': series.quantile(0.95), 'min': series.min(), 'max': series.max(), 'skewness': series.skew(), 'iqr_outliers': ((series < q1 - 1.5 * iqr) | (series > q3 + 1.5 * iqr)).sum()})
    return pd.DataFrame(rows).set_index('variable')


## 1. Coverage and data completeness


In [ ]:
coverage = district.groupby(['year', 'quarter'], as_index=False).agg(rows=('district', 'size'), states=('state', 'nunique'), districts=('district', 'nunique'), merchant_missing=('registered_merchants', lambda s: s.isna().sum()))
latest_missing = latest[['transaction_count', 'transaction_amount', 'registered_users', 'registered_merchants']].isna().sum()
display(coverage.tail(12))
display(latest_missing.rename('missing_values').to_frame())
insight(f'The latest period is {latest_label} with {len(latest):,} district rows across {latest.state.nunique():,} states/UTs.', f'Core latest-quarter missing values total {int(latest_missing.sum()):,}.', 'Historical merchant nulls are source limitations and should not be converted to zero.')


## 2. Univariate analysis and descriptive statistics


In [ ]:
core = ['transaction_count', 'transaction_amount', 'registered_users', 'registered_merchants', 'average_transaction_value', 'transactions_per_registered_user', 'tpv_per_registered_user', 'merchants_per_100k_users', 'users_per_merchant']
growth = ['transaction_count_qoq', 'transaction_amount_qoq', 'registered_users_qoq', 'registered_merchants_qoq', 'transaction_yoy']
display(profile(latest, core))
display(profile(latest, growth))
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
for ax, column in zip(axes.flat, core[:8] + ['transaction_yoy']):
    values = latest[column].replace([np.inf, -np.inf], np.nan).dropna()
    if column in {'transaction_count', 'transaction_amount', 'registered_users', 'registered_merchants'}:
        values = np.log1p(values)
        xlabel = f'log(1 + {column})'
    else:
        lower, upper = values.quantile([0.01, 0.99])
        values = values.clip(lower, upper)
        xlabel = column
    sns.histplot(values, bins=35, kde=True, ax=ax)
    ax.set_title(column.replace('_', ' ').title())
    ax.set_xlabel(xlabel)
save('univariate_distributions.png')
insight('District scale is strongly right-skewed, so medians and percentile ranks are more representative than means.', f"{latest['transaction_yoy'].gt(0).mean():.1%} of districts show positive same-quarter transaction growth.", 'Extreme growth observations should be checked for base effects before influencing prioritisation.')


## 3. State and district concentration


In [ ]:
state_rank = latest_state.sort_values('transaction_count', ascending=False).copy()
state_rank['transaction_share'] = state_rank['transaction_count'] / state_rank['transaction_count'].sum()
state_rank['cumulative_share'] = state_rank['transaction_share'].cumsum()
display(state_rank[['state', 'transaction_count', 'registered_users', 'registered_merchants', 'transaction_share', 'cumulative_share']].head(15))
top5_share = state_rank.head(5)['transaction_share'].sum()
top10_share = state_rank.head(10)['transaction_share'].sum()
plt.figure(figsize=(10, 6))
sns.barplot(data=state_rank.head(15), x='transaction_share', y='state')
plt.title(f'Top states by transaction share | {latest_label}')
save('state_transaction_concentration.png')
district_share = latest.copy()
district_share['share_of_state'] = district_share['transaction_count'] / district_share.groupby('state')['transaction_count'].transform('sum')
state_concentration = district_share.sort_values(['state', 'share_of_state'], ascending=[True, False]).groupby('state').head(3).groupby('state', as_index=False)['share_of_state'].sum().rename(columns={'share_of_state': 'top3_district_share'}).sort_values('top3_district_share', ascending=False)
display(state_concentration.head(15))
insight(f'The top 5 states account for {top5_share:.1%} of transactions and the top 10 account for {top10_share:.1%}.', 'Some states are dominated by a few districts, so statewide recommendations can hide local differences.', 'District-level prioritisation is especially important when transaction activity is concentrated in a small number of districts.')


## 4. Category composition and recent category trends


In [ ]:
latest_categories = categories[categories['period_id'].eq(categories['period_id'].max())]
mix = latest_categories.groupby('category', as_index=False)['transaction_count'].sum()
mix['share'] = mix['transaction_count'] / mix['transaction_count'].sum()
mix = mix.sort_values('share', ascending=False)
display(mix)
plt.figure(figsize=(9, 5))
sns.barplot(data=mix, x='share', y='category')
plt.title(f'Transaction category mix | {latest_label}')
save('transaction_category_mix.png')
category_history = categories.groupby(['period_id', 'year', 'quarter', 'category'], as_index=False)['transaction_count'].sum()
category_history['share'] = category_history['transaction_count'] / category_history.groupby('period_id')['transaction_count'].transform('sum')
recent_periods = sorted(category_history['period_id'].unique())[-8:]
recent_mix = category_history[category_history['period_id'].isin(recent_periods)].copy()
recent_mix['period'] = recent_mix['year'].astype(str) + ' Q' + recent_mix['quarter'].astype(str)
plt.figure(figsize=(12, 6))
sns.lineplot(data=recent_mix, x='period', y='share', hue='category', marker='o')
plt.xticks(rotation=45, ha='right')
plt.title('Transaction category share over the latest 8 quarters')
save('category_share_trend.png')
insight(f"{mix.iloc[0]['category']} is the largest latest-quarter category at {mix.iloc[0]['share']:.1%}.", 'Category mix matters because total PhonePe growth is not automatically merchant-payment growth.', 'District category data is unavailable, so district totals should be described as ecosystem activity rather than merchant-only activity.')


## 5. Top and bottom market diagnostics


In [ ]:
diagnostic_columns = ['state', 'district', 'transaction_count', 'registered_users', 'registered_merchants', 'transaction_yoy', 'merchants_per_100k_users', 'users_per_merchant']
display(Markdown('### Largest districts by transaction volume'))
display(latest.nlargest(15, 'transaction_count')[diagnostic_columns])
display(Markdown('### Fastest-growing sizeable districts'))
display(latest[latest['registered_users'].ge(100_000)].dropna(subset=['transaction_yoy']).nlargest(15, 'transaction_yoy')[diagnostic_columns])
display(Markdown('### Lowest merchant penetration among sizeable districts'))
display(latest[latest['registered_users'].ge(100_000) & latest['registered_merchants'].gt(0)].nsmallest(15, 'merchants_per_100k_users')[diagnostic_columns])
insight('The largest districts are not necessarily the fastest-growing or lowest-penetration markets.', 'Stronger expansion candidates appear across multiple screens: scale, growth, and relative merchant penetration.', 'Cross-checking multiple lists is more informative than ranking on one metric alone.')


## 6. Bivariate relationships


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
pair = latest.dropna(subset=['registered_users', 'transaction_count'])
sns.regplot(x=np.log1p(pair['registered_users']), y=np.log1p(pair['transaction_count']), scatter_kws={'alpha': 0.35, 's': 24}, ax=axes[0, 0])
axes[0, 0].set_title('Users vs transactions')
pair = latest.dropna(subset=['registered_merchants', 'transaction_count'])
sns.regplot(x=np.log1p(pair['registered_merchants']), y=np.log1p(pair['transaction_count']), scatter_kws={'alpha': 0.35, 's': 24}, ax=axes[0, 1])
axes[0, 1].set_title('Merchants vs transactions')
sns.scatterplot(data=latest, x='merchants_per_100k_users', y='transaction_yoy', size='registered_users', sizes=(15, 160), alpha=0.4, legend=False, ax=axes[1, 0])
axes[1, 0].axhline(0, linestyle='--', linewidth=1)
axes[1, 0].set_title('Growth vs merchant penetration')
sns.scatterplot(data=latest, x='registered_users_qoq', y='registered_merchants_qoq', size='transaction_count', sizes=(15, 160), alpha=0.4, legend=False, ax=axes[1, 1])
axes[1, 1].axline((0, 0), slope=1, linestyle='--', linewidth=1)
axes[1, 1].set_title('User growth vs merchant growth')
save('bivariate_relationships.png')
rho_users = latest[['registered_users', 'transaction_count']].corr(method='spearman').iloc[0, 1]
rho_merchants = latest[['registered_merchants', 'transaction_count']].corr(method='spearman').iloc[0, 1]
rho_penetration = latest[['merchants_per_100k_users', 'transaction_yoy']].corr(method='spearman').iloc[0, 1]
growth_gap_share = (latest['registered_users_qoq'] > latest['registered_merchants_qoq']).mean()
insight(f'Registered users and transactions have Spearman rho {rho_users:.3f}.', f'Registered merchants and transactions have Spearman rho {rho_merchants:.3f}.', f'Merchant penetration and YoY growth have Spearman rho {rho_penetration:.3f}.', f'User growth exceeds merchant growth in {growth_gap_share:.1%} of latest-quarter districts.', 'These are descriptive associations, not causal effects.')


## 7. Merchant-penetration cohorts and multivariate opportunity quadrants


In [ ]:
cohort = latest.dropna(subset=['merchants_per_100k_users', 'transaction_yoy', 'transaction_count', 'registered_users']).copy()
cohort['penetration_quartile'] = pd.qcut(cohort['merchants_per_100k_users'], 4, labels=['Q1 lowest', 'Q2', 'Q3', 'Q4 highest'], duplicates='drop')
cohort_summary = cohort.groupby('penetration_quartile', observed=False).agg(districts=('district', 'size'), median_transactions=('transaction_count', 'median'), median_users=('registered_users', 'median'), median_yoy=('transaction_yoy', 'median'), median_merchants_per_100k=('merchants_per_100k_users', 'median')).reset_index()
display(cohort_summary)
plt.figure(figsize=(9, 5))
sns.boxplot(data=cohort, x='penetration_quartile', y='transaction_yoy')
plt.title('Transaction growth by merchant-penetration quartile')
save('growth_by_penetration_quartile.png')
growth_cut = cohort['transaction_yoy'].median()
penetration_cut = cohort['merchants_per_100k_users'].median()
cohort['quadrant'] = np.select([(cohort['transaction_yoy'] >= growth_cut) & (cohort['merchants_per_100k_users'] < penetration_cut), (cohort['transaction_yoy'] >= growth_cut) & (cohort['merchants_per_100k_users'] >= penetration_cut), (cohort['transaction_yoy'] < growth_cut) & (cohort['merchants_per_100k_users'] < penetration_cut)], ['High growth / low penetration', 'High growth / high penetration', 'Low growth / low penetration'], default='Low growth / high penetration')
display(cohort.groupby('quadrant').agg(districts=('district', 'size'), median_transactions=('transaction_count', 'median'), median_users=('registered_users', 'median'), median_yoy=('transaction_yoy', 'median'), median_penetration=('merchants_per_100k_users', 'median')).reset_index())
plt.figure(figsize=(11, 7))
sns.scatterplot(data=cohort, x='merchants_per_100k_users', y='transaction_yoy', size='registered_users', hue='quadrant', sizes=(20, 240), alpha=0.55)
plt.axvline(penetration_cut, linestyle='--', linewidth=1)
plt.axhline(growth_cut, linestyle='--', linewidth=1)
plt.title('District opportunity quadrants')
save('opportunity_quadrants.png')
high_low = cohort[cohort['quadrant'].eq('High growth / low penetration')]
insight(f'{len(high_low):,} districts fall in the high-growth / low-penetration quadrant using median cutoffs.', 'Penetration quartiles are easier to interpret than one arbitrary national threshold.', 'Low penetration should only be considered an opportunity signal when demand, scale, and growth are also healthy.')


## 8. Multivariate correlation and score-input redundancy


In [ ]:
corr_columns = ['transaction_count', 'transaction_amount', 'registered_users', 'registered_merchants', 'average_transaction_value', 'transactions_per_registered_user', 'tpv_per_registered_user', 'merchants_per_100k_users', 'users_per_merchant', 'transaction_yoy', 'transaction_count_qoq', 'registered_users_qoq', 'registered_merchants_qoq']
corr = latest[corr_columns].corr(method='spearman')
plt.figure(figsize=(12, 9))
sns.heatmap(corr, cmap='vlag', center=0, annot=True, fmt='.2f')
plt.title(f'Spearman correlation matrix | {latest_label}')
save('correlation_heatmap.png')
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
strong_pairs = upper.stack().rename('rho').reset_index().assign(abs_rho=lambda df: df['rho'].abs()).sort_values('abs_rho', ascending=False).head(10)
display(strong_pairs)
insight('Strongly correlated inputs should not receive separate large weights because that can double-count the same market characteristic.', 'The correlation matrix is used as both EDA and a control on opportunity-score design.', 'Transactions per merchant should be treated as an ecosystem-intensity proxy, not direct merchant sales.')


## 9. Growth consistency and volatility


In [ ]:
recent_periods = sorted(district['period_id'].unique())[-4:]
recent4 = district[district['period_id'].isin(recent_periods)]
consistency = recent4.groupby(['state', 'district'], as_index=False).agg(positive_yoy_quarters=('transaction_yoy', lambda s: int(s.gt(0).sum())), average_yoy=('transaction_yoy', 'mean'), yoy_std=('transaction_yoy', 'std'), average_qoq=('transaction_count_qoq', 'mean'))
latest_consistency = latest.merge(consistency, on=['state', 'district'], how='left')
stable_growth = latest_consistency.query('positive_yoy_quarters == 4').copy()
display(stable_growth[['state', 'district', 'transaction_count', 'registered_users', 'positive_yoy_quarters', 'average_yoy', 'yoy_std']].sort_values(['average_yoy', 'transaction_count'], ascending=[False, False]).head(20))
volatility = district.dropna(subset=['transaction_yoy']).groupby(['state', 'district'], as_index=False).agg(yoy_mean=('transaction_yoy', 'mean'), yoy_std=('transaction_yoy', 'std'), observed_quarters=('transaction_yoy', 'count'))
latest_volatility = latest.merge(volatility, on=['state', 'district'], how='left')
eligible_volatility = latest_volatility[latest_volatility['registered_users'].ge(100_000) & latest_volatility['observed_quarters'].ge(8)]
plt.figure(figsize=(10, 6))
sns.scatterplot(data=eligible_volatility, x='yoy_mean', y='yoy_std', size='registered_users', sizes=(15, 160), alpha=0.4, legend=False)
plt.title('Average growth vs growth volatility')
save('growth_volatility.png')
insight(f'{len(stable_growth):,} districts recorded positive YoY growth in each of the latest four quarters.', 'Persistent growth is more reliable for prioritisation than one exceptional quarter.', 'High growth with high volatility deserves more validation than similar growth with stable history.')


## 10. National indexed growth and intensity


In [ ]:
national = state.groupby(['period_id', 'year', 'quarter'], as_index=False).agg(transaction_count=('transaction_count', 'sum'), transaction_amount=('transaction_amount', 'sum'), registered_users=('registered_users', 'sum'), registered_merchants=('registered_merchants', 'sum')).sort_values('period_id')
for column in ['transaction_count', 'transaction_amount', 'registered_users', 'registered_merchants']:
    base = national[column].dropna().iloc[0]
    national[f'{column}_index'] = 100 * national[column] / base
national['period'] = national['year'].astype(str) + ' Q' + national['quarter'].astype(str)
indexed = national.melt(id_vars=['period'], value_vars=['transaction_count_index', 'transaction_amount_index', 'registered_users_index', 'registered_merchants_index'], var_name='metric', value_name='index')
plt.figure(figsize=(12, 6))
sns.lineplot(data=indexed, x='period', y='index', hue='metric')
plt.xticks(rotation=45, ha='right')
plt.title('Indexed national growth since first available period')
save('national_indexed_growth.png')
display(national.iloc[-1][['transaction_count_index', 'transaction_amount_index', 'registered_users_index', 'registered_merchants_index']].to_frame('latest_index'))
insight('Indexed growth shows whether payment activity is expanding faster or slower than the registered-user and merchant bases.', 'If transactions outpace registrations, ecosystem usage intensity is increasing.', 'This comparison is directional because cumulative registrations do not measure active monthly participants.')


## 11. Outlier review


In [ ]:
outlier_rows = []
for column in core + growth:
    series = latest[column].replace([np.inf, -np.inf], np.nan).dropna()
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    flags = latest[column].lt(lower) | latest[column].gt(upper)
    if flags.any():
        temp = latest.loc[flags, ['state', 'district', column]].copy()
        temp['metric'] = column
        temp['lower_fence'] = lower
        temp['upper_fence'] = upper
        outlier_rows.append(temp.rename(columns={column: 'value'}))
outliers = pd.concat(outlier_rows, ignore_index=True)
display(outliers.groupby('metric').size().rename('outlier_count').sort_values(ascending=False).to_frame())
display(outliers.sort_values('value', ascending=False).head(25))
insight('Outliers are not automatically errors; major metros and newly scaling districts can be legitimate extremes.', 'The review is used to stop extreme raw values from dominating averages, correlations, or score weights.', 'Percentile ranks and robust comparisons are more appropriate than raw-value weighting for this project.')


## 12. EDA conclusion


In [ ]:
summary = pd.DataFrame({'Question': ['Where is activity concentrated?', 'Where is growth strongest?', 'Where is merchant penetration relatively low?', 'Which markets show persistent growth?', 'Which markets are volatile?', 'What remains unknown?'], 'EDA answer': [f'Top 5 states contribute {top5_share:.1%} of latest-quarter transactions.', f"{latest['transaction_yoy'].gt(0).mean():.1%} of districts have positive YoY growth.", f'{len(high_low):,} districts are high-growth / low-penetration using median cutoffs.', f'{len(stable_growth):,} districts posted positive YoY growth in all latest four quarters.', 'Historical growth volatility varies materially across districts.', 'Active merchant acceptance, competition, acquisition cost, district category mix, and merchant activity are not available in Pulse.']})
display(summary)
insight('EDA supports a prioritisation workflow, not a causal expansion claim.', 'The strongest candidates are sizeable districts combining sustained growth, relatively low merchant penetration, and reasonable stability.', 'Field validation remains necessary because registered merchants are not active merchants and district transactions are not merchant-only transactions.')
